In [ ]:
!wget -O image.jpg #"Write here each image link"

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Objective Function: Otsu's Between-Class Variance
def otsu_fitness(thresholds, image):
    thresholds = np.sort(np.array(thresholds, dtype=np.uint8))
    bins = np.histogram(image, bins=256, range=(0, 256))[0]
    total_pixels = image.size
    thresholds = np.concatenate([[0], thresholds, [256]])

    variance = 0
    for i in range(len(thresholds) - 1):
        start, end = thresholds[i], thresholds[i+1]
        hist_range = bins[start:end]
        prob = np.sum(hist_range) / total_pixels
        if prob == 0:
            continue
        mean = np.sum(np.arange(start, end) * hist_range) / np.sum(hist_range)
        total_mean = np.mean(image)
        variance += prob * ((mean - total_mean) ** 2)
    return variance

# Gray Wolf Optimizer Implemetation
def gwo(fitness_func, dim, bounds, n_agents=20, max_iter=50, image=None):
    alpha = np.zeros(dim)
    beta = np.zeros(dim)
    delta = np.zeros(dim)
    alpha_score = -np.inf
    beta_score = -np.inf
    delta_score = -np.inf

# Initialize the search agents
    positions = np.random.randint(bounds[0], bounds[1], size=(n_agents, dim))

    for t in range(max_iter):
        for i in range(n_agents):
            fitness = fitness_func(positions[i], image)
            if fitness > alpha_score:
                delta_score, delta = beta_score, beta.copy()
                beta_score, beta = alpha_score, alpha.copy()
                alpha_score, alpha = fitness, positions[i].copy()
            elif fitness > beta_score:
                delta_score, delta = beta_score, beta.copy()
                beta_score, beta = fitness, positions[i].copy()
            elif fitness > delta_score:
                delta_score, delta = fitness, positions[i].copy()

        a = 2 - t * (2 / max_iter)
        for i in range(n_agents):
            for j in range(dim):
                r1, r2 = np.random.rand(), np.random.rand()
                A1, C1 = 2*a*r1 - a, 2*r2
                D_alpha = abs(C1*alpha[j] - positions[i][j])
                X1 = alpha[j] - A1*D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2, C2 = 2*a*r1 - a, 2*r2
                D_beta = abs(C2*beta[j] - positions[i][j])
                X2 = beta[j] - A2*D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3, C3 = 2*a*r1 - a, 2*r2
                D_delta = abs(C3*delta[j] - positions[i][j])
                X3 = delta[j] - A3*D_delta

                new_val = (X1 + X2 + X3) / 3
                new_val = np.clip(new_val, bounds[0], bounds[1])
                positions[i][j] = int(new_val)

    return np.sort(alpha.astype(np.uint8)), alpha_score

# Load grayscale image
img = cv2.imread("image.jpg", cv2.IMREAD_GRAYSCALE)
print(img.shape)

# Run GWO for 5 thresholds
best_thresholds, best_score = gwo(
    fitness_func=otsu_fitness,
    dim=5, ####################################################Change this
    bounds=(0, 255),
    n_agents=25,
    max_iter=80,
    image=img
)

# Apply thresholds
thresholded_img = np.zeros_like(img)
t = np.concatenate(([0], best_thresholds, [256]))
for i in range(len(t) - 1):
    mask = (img >= t[i]) & (img < t[i+1])
    thresholded_img[mask] = int((t[i] + t[i+1]) / 2)

# Show result
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title("Original")
plt.imshow(img, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Segmented using GWO (5 Thresholds)")
plt.imshow(thresholded_img, cmap='gray')
plt.axis('off')
plt.show()

print("Best thresholds:", best_thresholds)